In [2]:
import numpy as np
import networkx as nx
import pandas as pd
import geopandas as gpd
import osmnx as ox
import random

from genesis.core import BestPointsBase, MetricBase, StateBase
from genesis.metrics import ArrivalTime
from genesis.states import FirstArrivalUnitState
from genesis.tools import list_dict_concat

In [57]:
# Загрузка графа и полигона границ
G = ox.load_graphml("tests/data/test_rng.ml")
area = gpd.read_file("tests/data/test_polygon.gpkg")

In [139]:



class BestNodesGA(BestPointsBase):
    '''
    Поиск лучших узлов для размещения n объектов.

    Расчет производится генетическим алгоритмом.

    ## Область применения
    Определение размещения множества узлов в графе, при котором 
    обеспечиваются наилучшие некоторые целевые метрики. 
    
    ? Дает достаточно точное решение. Хорошо подходит для больших графов.
    '''

    def __init__(self,
                 state_function: StateBase,
                 metric_function: MetricBase,
                 population_size=25,
                 epochs:int=50,
                 mutation_rate=0.1,
                 appr_val_in_area=0,
                 **kwargs) -> None:
        '''
        ## Аргументы
        `state_function`: StateBase
            функция расчета состояния окружения
        `metric_function`: MetricBase
            Функция расчета метрики
        `best_point_function`: BestPointsBase
            Функция расчета лучшего размещения узла
        `iterations`:int=5
            Количество итераций расчета
        `appr_val_in_area`: int=0
            Доля узлов графа в пределах area, покрытие которой считается приемлемой для принятия расчетной метрики. 
            Если при расчете метрик, из стартового узла (узлов) достижимо меньшее количество узлов,
            то такой узел не рассматривается.
            При расчет размещения нескольких узлов неминуемо возникает ситуация при которой
            часть территории area будет недостижима. Поэтому по умолчанию считается
        '''
        
        self.population_size = population_size
        self.epochs = epochs
        self.mutation_rate = mutation_rate
        self.appr_val_in_area = appr_val_in_area
        super().__init__(state_function, metric_function, **kwargs)

    def _fit_function(self, env, nodes, area=None, **kwargs):
        '''
        Расчет функции приспособленности
        '''
        # 2.1. Расчет состояния
        times, _ = self.state_function(env=env, points=nodes, **kwargs)

        # 2.2. Расчет стартовой метрики состояния и определение стартового размещения
        best_metric = self.metric_function(times, area=area, **kwargs)

        return best_metric


    def __call__(self,
                 env:nx.MultiDiGraph,
                 dynamic_nodes: list|dict,
                 static_nodes: list|dict = None,
                 before_iters_start_function: callable =None,
                 iter_calc_end_function: callable =None,
                 area: pd.Series = None,
                 **kwargs):
        
        # 0. Проверка корректности пришедших данных
        if not isinstance(env, nx.MultiDiGraph):
            raise TypeError("Тип аргумента `env` должен быть MultiDiGraph!")
        if not static_nodes is None:
            if not ((isinstance(dynamic_nodes,list) and isinstance(static_nodes,list)) or \
                (isinstance(dynamic_nodes,dict) and isinstance(static_nodes,dict))):
                raise TypeError(f'Аргументы `dynamic_nodes` и `static_nodes` должны быть одинакового типа: list|dict'
                                f'Имеют: {type(dynamic_nodes)}, {type(static_nodes)}')
        if len(dynamic_nodes)<2:
            raise ValueError(f'Количество элементов `dynamic_nodes` не может быть меньше 2. Сейчас {len(dynamic_nodes)}')
        if not area is None and not isinstance(area, pd.Series):
            raise TypeError(f'Аргумент `area` должен иметь тип `pd.Series`! Имеет {type(area)}')

        # Если передана область которую следует учитывать в расчете
        # установить для нее приемлемый охват равный `self.appr_val_in_area`
        # (0 по умолчанию)
        if not area is None:
            self.best_point_function.__setattr__('appr_val',
                                                 self.appr_val_in_area)

        # 1.1. Если статические узлы не указаны - заменяем значение переменной с None
        # на [] или {} в зависимости от типа данных `dynamic_nodes`
        if static_nodes is None:
            if isinstance(dynamic_nodes,list):
                static_nodes = []
            elif isinstance(dynamic_nodes,dict):
                static_nodes = {}

        # Последовательность узлов графа (для последующего обращения к нему)
        g_nodes = list(env.nodes())

        # # 1.2. Получение суммарного списка (или словаря) узлов для расчета состояния и метрик
        # start_nodes = list_dict_concat(dynamic_nodes, static_nodes)

        # ===================================== Генетический алгоритм ==============
        # 1. Создание стартовой популяции
        population = [dynamic_nodes for _ in range(self.population_size)]
        
        # 2. На каждой эпохе
        for epoch in range(self.epochs):
            # print(epoch, population)
            # Оценка приспособленности всех особей
            bot_fit = [self._fit_function(env=env, nodes=list_dict_concat(dn, static_nodes)) for dn in population]
            # print(epoch, bot_fit, min(bot_fit))
            print(epoch, min(bot_fit))

            new_population = []
            # Генерация новой популяции
            for _ in range(self.population_size):
                # Отбор по правилу рулетки
                parent_bot_1 = random.choices(population, weights=bot_fit)[0]
                parent_bot_2 = random.choices(population, weights=bot_fit)[0]

                # Скрещивание (одноточечное)
                split_point = int(len(parent_bot_1)/2)
                left_gen_vals = list(parent_bot_1.values())[:split_point]
                right_gen_vals  = list(parent_bot_1.values())[split_point:]

                # print(parent_bot_1, parent_bot_2, left_gen_vals, right_gen_vals)
                left_genome_part = {k:v for k, v in parent_bot_1.items() if v in left_gen_vals}
                right_genome_part = {k:v for k, v in parent_bot_2.items() if v in right_gen_vals}

                new_dynamic_nodes = {**left_genome_part, **right_genome_part}

                # Мутация (выбор произвольного узла)
                if random.random() < self.mutation_rate:
                    # Выбор случайного элемента в словаре и удаление его из new_dynamic_nodes
                    node, unit = random.choice(list(new_dynamic_nodes.items()))
                    del new_dynamic_nodes[node]
                    # print(node, unit, new_dynamic_nodes)

                    # Поиск нового узла, котрого при этом нет в new_dynamic_nodes
                    node = random.choice(g_nodes)
                    while node in new_dynamic_nodes.keys():
                        node = random.choice(g_nodes)

                    new_dynamic_nodes[node] = unit

                    # print(node, unit, new_dynamic_nodes)

                # Добавляем его в новую популяцию
                new_population.append(new_dynamic_nodes)

            # Заменяем предыдущую популяцию новой
            population = new_population
            # print(population)

        return bot_fit

nodes = list(G.nodes())
existed_units = {nodes[100]:'A', 
                nodes[2000]:'B', 
                }
new_units = {nodes[3000]:'c',
             nodes[4000]:'d'}


BNGA = BestNodesGA(FirstArrivalUnitState(), ArrivalTime(), population_size=50, epochs=200, mutation_rate=0.9)


BNGA(env=G, static_nodes=existed_units, dynamic_nodes=new_units)



0 7.262001771503841
1 5.571123859232247
2 5.443110593749999
3 5.126034589454423
4 5.171450607194768
5 5.165359462949024
6 5.650525662907496
7 5.2401887112100285
8 4.990887788024294
9 5.358961831161753
10 5.731944001946636
11 5.272935257851433
12 5.6564750802663
13 5.744389104456499
14 5.451995078527305
15 5.416744149462727
16 5.451826892039555
17 5.453808976276994
18 5.355181405886628
19 5.586603717504153


[7.157931175236193,
 6.717919993264638,
 6.095658222266923,
 5.858213152707122,
 5.89254974443262,
 6.101107009953799,
 6.731423860023878,
 6.25018262040594,
 7.3467716382371275,
 6.529430883967504,
 7.781485524358908,
 6.332746368134863,
 7.7064178876142035,
 7.2965689724486085,
 7.353518327229548,
 7.787492881709405,
 7.384748502699336,
 6.063206871820493,
 5.636316398787894,
 7.5296715604884765,
 7.741329883630088,
 6.339914762588248,
 6.571941218256852,
 6.181335225524295,
 7.854793818054402,
 8.106112866992836,
 6.4916188602964064,
 7.492585264508929,
 5.81691597394103,
 6.351300143570909,
 6.325744081293752,
 7.58519126605326,
 6.614811544941341,
 7.264050993420369,
 6.929236387601224,
 5.79396627975758,
 6.963424580863269,
 6.031231509330877,
 6.709224994899812,
 7.042827114864514,
 6.583665191432204,
 7.551798043643584,
 7.2417337349589905,
 6.230446764729547,
 6.940905628620745,
 5.9742949980144315,
 5.819060972318833,
 5.586603717504153,
 7.563640462507786,
 6.185414985036856

In [122]:
parent_bot_1 = {2760591335: 'c', 4901942432: 'd'}
split_point=int(len(parent_bot_1)/2)
{k:v for i, (k, v) in enumerate(parent_bot_1.items()) if i<split_point}

{2760591335: 'c'}

In [62]:
a = [1,2,3,4,5,6,7]
w = [1,2,1,1,3,2,1]
p = [x/sum(w) for x in w]
np.random.choice(a, size=10, p=p)

array([5, 2, 1, 2, 6, 2, 5, 6, 5, 6])

In [132]:
d0 = {1:'1', 2:'2', 3:'3'}
p = 1
d1 = {k:v for i, (k, v) in enumerate(d0.items()) if i<p}
print(d1)
d2 = {k:v for i, (k, v) in enumerate(d0.items()) if i>=p}
print(d2)

d3 = {**d1, **d2}
print(d3)

{1: '1'}
{2: '2', 3: '3'}
{1: '1', 2: '2', 3: '3'}


In [134]:
d3[1,2]

KeyError: (1, 2)

In [107]:
k,v = random.choice(list(d3.items()))
new_element = {k:v}
print(new_element)

del d3[k]
print(d3)

{1: '1'}
{2: '2', 3: '3'}


In [103]:
# словарь
my_dict = {'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5}

# удаление и получение последнего добавленного элемента
key, value = my_dict.popitem()

print(my_dict)
print(key, value)

{'a': 1, 'b': 2, 'c': 3, 'd': 4}
e 5


In [42]:
def f1(a, w):
    return np.random.choice(a, size=10, p=pd.Series(w)/sum(w))

def f2(a, w):
    return random.choices(a, weights=w, k=10)

print(f1(a, w))
print(f2(a, w))

[7, 5, 2, 5, 6, 2, 2, 2, 7, 6]
[2, 7, 2, 5, 5, 2, 2, 6, 2, 2]


In [47]:
random.choices(a, weights=w)[0]

5

In [56]:
set1 = set([1,2,3])
set2 = set([3,4,5,6])
v1 = set([2])
# set1-set2
set1-v1

{1, 3}

In [45]:
%timeit f1(a, w)

333 µs ± 3.85 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [44]:
%timeit f2(a, w)

7.42 µs ± 149 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
